# Notebook 5 — DFC com Hierarquia

**Objetivo:** Extrair a DFC da Silver consolidada, reconstruir a hierarquia em Safe Mode, validar a equação fundamental (6.01 + 6.02 + 6.03 ≈ 6.05.02) com tolerância de 1% e escrever a tabela curada.  
**Input:** `layer_02_silver.n0_dfp_cia_aberta`  
**Output:** `layer_02_silver.n1_dfp_cia_aberta_dfc`

## 1. Importando Bibliotecas

In [2]:
import pandas as pd
import datetime
import logging
import os
import numpy as np
from sqlalchemy import create_engine, text, inspect
from urllib.parse import quote_plus
from dotenv import load_dotenv

In [3]:
# Carrega variáveis de ambiente
load_dotenv()

True

In [4]:
# Definindo a função para criar a engine do banco de dados
def create_db_engine():
        user = quote_plus(os.getenv('DB_USER', 'postgres'))
        password = quote_plus(os.getenv('DB_PASS', 'password'))
        host = os.getenv('DB_HOST', 'localhost')
        port = os.getenv('DB_PORT', '5432')
        dbname = os.getenv('DB_NAME', 'data_lake')
        
        connection_str = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}"
        return create_engine(connection_str)

In [5]:
# Efetivamente criando a engine
engine = create_db_engine()

## 2. Query de Extração da DFC — deduplicação por versão

> Parte de `n0_dfp_cia_aberta` filtrando apenas contas de fluxo de caixa (GRUPO_DFP LIKE '%Fluxo%').  
> O `ROW_NUMBER()` garante apenas a versão mais recente por empresa/período/conta.

In [6]:
query_dfc_hierarquias = '''
WITH base_bruta AS (
    SELECT 
        *,
        -- Cria um ranking para pegar a versão mais recente, mas preserva duplicatas legítimas de conta
        -- se o valor ou a descrição forem diferentes.
        ROW_NUMBER() OVER(
            PARTITION BY "CNPJ_CIA", "DT_REFER", "CD_CONTA", "DS_CONTA", "VL_CONTA_TRATADO"
            ORDER BY "VERSAO" DESC
        ) as rn
    FROM layer_02_silver.n0_dfp_cia_aberta
    WHERE "GRUPO_DFP_TRATADO" = 'DFC'
),
base_inicial AS (
    SELECT 
        "CNPJ_CIA" || "DT_REFER" || "GRUPO_DFP_TRATADO" || left(REPLACE("CD_CONTA", '.',''), 1) as "CHAVE_BUSCA_NIVEL_1",
        "CNPJ_CIA" || "DT_REFER" || "GRUPO_DFP_TRATADO" || left(REPLACE("CD_CONTA", '.',''), 3) as "CHAVE_BUSCA_NIVEL_2",
        "CNPJ_CIA" || "DT_REFER" || "GRUPO_DFP_TRATADO" || left(REPLACE("CD_CONTA", '.',''), 5) as "CHAVE_BUSCA_NIVEL_3",
        "CNPJ_CIA" || "DT_REFER" || "GRUPO_DFP_TRATADO" || left(REPLACE("CD_CONTA", '.',''), 7) as "CHAVE_BUSCA_NIVEL_4",
        "CNPJ_CIA" || "DT_REFER" || "GRUPO_DFP_TRATADO" || left(REPLACE("CD_CONTA", '.',''), 9) as "CHAVE_BUSCA_NIVEL_5",
        t.*
    FROM base_bruta t
    WHERE rn = 1
),

-- Tabelas de Referência (Mantemos DISTINCT ON aqui para pegar a descrição canônica de cada nível)
ref_nivel_2 as (
    SELECT DISTINCT ON ("CHAVE_MATCH")
        "CNPJ_CIA" || "DT_REFER" || "GRUPO_DFP_TRATADO" || left(REPLACE("CD_CONTA", '.',''), 3) as "CHAVE_MATCH",
        UPPER(TRIM("DS_CONTA")) as "DS_NIVEL_2"
    FROM layer_02_silver.n0_dfp_cia_aberta 
    WHERE "GRUPO_DFP_TRATADO" = 'DFC' AND "CD_CONTA_QTD_DIGITOS" = 3
    ORDER BY "CHAVE_MATCH", "VERSAO" DESC
),
ref_nivel_3 as (
    SELECT DISTINCT ON ("CHAVE_MATCH")
        "CNPJ_CIA" || "DT_REFER" || "GRUPO_DFP_TRATADO" || left(REPLACE("CD_CONTA", '.',''), 5) as "CHAVE_MATCH",
        UPPER(TRIM("DS_CONTA")) as "DS_NIVEL_3"
    FROM layer_02_silver.n0_dfp_cia_aberta 
    WHERE "GRUPO_DFP_TRATADO" = 'DFC' AND "CD_CONTA_QTD_DIGITOS" = 5
    ORDER BY "CHAVE_MATCH", "VERSAO" DESC
),
ref_nivel_4 as (
    SELECT DISTINCT ON ("CHAVE_MATCH")
        "CNPJ_CIA" || "DT_REFER" || "GRUPO_DFP_TRATADO" || left(REPLACE("CD_CONTA", '.',''), 7) as "CHAVE_MATCH",
        UPPER(TRIM("DS_CONTA")) as "DS_NIVEL_4"
    FROM layer_02_silver.n0_dfp_cia_aberta 
    WHERE "GRUPO_DFP_TRATADO" = 'DFC' AND "CD_CONTA_QTD_DIGITOS" = 7
    ORDER BY "CHAVE_MATCH", "VERSAO" DESC
),
ref_nivel_5 as (
    SELECT DISTINCT ON ("CHAVE_MATCH")
        "CNPJ_CIA" || "DT_REFER" || "GRUPO_DFP_TRATADO" || left(REPLACE("CD_CONTA", '.',''), 9) as "CHAVE_MATCH",
        UPPER(TRIM("DS_CONTA")) as "DS_NIVEL_5"
    FROM layer_02_silver.n0_dfp_cia_aberta 
    WHERE "GRUPO_DFP_TRATADO" = 'DFC' AND "CD_CONTA_QTD_DIGITOS" = 9
    ORDER BY "CHAVE_MATCH", "VERSAO" DESC
)

-- JOIN FINAL
SELECT 
    base."CNPJ_CIA",
    empresas_selecionadas."SETOR_ATIV",
    base."DT_REFER",
    base."DT_REFER_TRATADO", 
    base."DT_REFER_ANO", 
    base."VERSAO", 
    base."DENOM_CIA", 
    base."CD_CVM",  
    base."GRUPO_DFP_TRATADO", 
    base."DT_FIM_EXERC_TRATADO", 
    base."DT_FIM_EXERC_ANO", 
    base."CD_CONTA", 
    base."CD_CONTA_QTD_DIGITOS", 
    base."DS_CONTA",
    -- Criação da coluna composta para facilitar leitura (Ex: "6.01 - Caixa Operacional")
    base."CD_CONTA" || ' - ' || base."DS_CONTA" as "CONTA_NOME_COMPLETO",
    base."VL_CONTA_TRATADO", 
    base."ST_CONTA_FIXA", 
    base."IS_LEAF",
    
    -- Como DFC não tem conta raiz "6", forçamos o nome do Nível 1
    'DEMONSTRAÇÃO DO FLUXO DE CAIXA' as "DS_NIVEL_1",
    n2."DS_NIVEL_2",
    n3."DS_NIVEL_3",
    n4."DS_NIVEL_4",
    n5."DS_NIVEL_5",
    
    'layer_02_silver.n0_dfp_cia_aberta' as "_origem_tabela"
FROM base_inicial base
LEFT JOIN ref_nivel_2 n2 ON base."CHAVE_BUSCA_NIVEL_2" = n2."CHAVE_MATCH"
LEFT JOIN ref_nivel_3 n3 ON base."CHAVE_BUSCA_NIVEL_3" = n3."CHAVE_MATCH"
LEFT JOIN ref_nivel_4 n4 ON base."CHAVE_BUSCA_NIVEL_4" = n4."CHAVE_MATCH"
LEFT JOIN ref_nivel_5 n5 ON base."CHAVE_BUSCA_NIVEL_5" = n5."CHAVE_MATCH"
LEFT JOIN layer_02_silver.n0_empresas_selecionadas as empresas_selecionadas
            ON base."CNPJ_CIA" = empresas_selecionadas."CNPJ_CIA"
ORDER BY base."CNPJ_CIA", base."DT_REFER", base."CD_CONTA";
'''

## 3. Extração dos Dados

In [7]:
# 1. Extração e Consolidação em Memória
with engine.connect() as conn:
    df_dfc = pd.read_sql(
        text(query_dfc_hierarquias), 
        con=conn
    )

In [8]:
df_dfc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88876 entries, 0 to 88875
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   CNPJ_CIA              88876 non-null  object 
 1   SETOR_ATIV            88876 non-null  object 
 2   DT_REFER              88876 non-null  object 
 3   DT_REFER_TRATADO      88876 non-null  object 
 4   DT_REFER_ANO          88876 non-null  float64
 5   VERSAO                88876 non-null  object 
 6   DENOM_CIA             88876 non-null  object 
 7   CD_CVM                88876 non-null  object 
 8   GRUPO_DFP_TRATADO     88876 non-null  object 
 9   DT_FIM_EXERC_TRATADO  88876 non-null  object 
 10  DT_FIM_EXERC_ANO      88876 non-null  float64
 11  CD_CONTA              88876 non-null  object 
 12  CD_CONTA_QTD_DIGITOS  88876 non-null  int64  
 13  DS_CONTA              88876 non-null  object 
 14  CONTA_NOME_COMPLETO   88876 non-null  object 
 15  VL_CONTA_TRATADO   

In [9]:
df_dfc.columns

Index(['CNPJ_CIA', 'SETOR_ATIV', 'DT_REFER', 'DT_REFER_TRATADO',
       'DT_REFER_ANO', 'VERSAO', 'DENOM_CIA', 'CD_CVM', 'GRUPO_DFP_TRATADO',
       'DT_FIM_EXERC_TRATADO', 'DT_FIM_EXERC_ANO', 'CD_CONTA',
       'CD_CONTA_QTD_DIGITOS', 'DS_CONTA', 'CONTA_NOME_COMPLETO',
       'VL_CONTA_TRATADO', 'ST_CONTA_FIXA', 'IS_LEAF', 'DS_NIVEL_1',
       'DS_NIVEL_2', 'DS_NIVEL_3', 'DS_NIVEL_4', 'DS_NIVEL_5',
       '_origem_tabela'],
      dtype='object')

## 4. Pipeline Final — cura e validação da DFC

### 4.1 Declaração das Funções — Fase 1 (Safe Mode) e Fase 2 (Validação Horizontal)

> **Fase 1:** reconstrói a hierarquia preenchendo apenas lacunas (nunca sobrescreve valores originais).  
> **Fase 2:** valida a equação `6.01 + 6.02 + 6.03 ≈ 6.05.02` com tolerância de 1% (CPC 03 §32).

In [10]:
# ==============================================================================
# 1. CURA E RECONSTRUÇÃO VERTICAL (V11 - DFC SAFE MODE)
# ==============================================================================
def executar_sanity_check_v11_dfc_safe(df_input):
    print("🏗️ INICIANDO SANITY CHECK V11 (DFC SAFE MODE - VERTICAL CHECK)...")
    
    # 1. Deduplicação e Limpeza Inicial
    # Agrupa por chave primária para garantir unicidade antes da matemática
    cols_group = ['CNPJ_CIA', 'DT_REFER', 'CD_CONTA', 'CD_CONTA_QTD_DIGITOS']
    df = df_input.groupby(cols_group).agg({'VL_CONTA_TRATADO': 'sum', 'DENOM_CIA': 'first'}).reset_index()
    
    # Ordena níveis do mais profundo para o mais raso
    niveis = sorted(df['CD_CONTA_QTD_DIGITOS'].unique(), reverse=True)
    total_reconstruido = 0
    total_protegido = 0
    
    for nivel_filho in niveis:
        # A DFC começa "de verdade" no nível 3 (ex: 6.01). Não existe conta "6" agregadora.
        # Então paramos o loop se o nível filho for 3, pois o pai seria "6" (que não queremos criar)
        if nivel_filho <= 3: 
            continue 
            
        df_filhos = df[df['CD_CONTA_QTD_DIGITOS'] == nivel_filho].copy()
        if df_filhos.empty: continue

        # Identifica Pai Virtual (ex: 6.01.01.04 -> pai 6.01.01)
        df_filhos['CD_PAI_VIRTUAL'] = df_filhos['CD_CONTA'].astype(str).str.rsplit('.', n=1).str[0]
        
        # Soma Filhos
        soma_filhos = df_filhos.groupby(['CNPJ_CIA', 'DT_REFER', 'CD_PAI_VIRTUAL']).agg({
            'VL_CONTA_TRATADO': 'sum', 'DENOM_CIA': 'first'
        }).reset_index()
        
        soma_filhos.rename(columns={'VL_CONTA_TRATADO': 'VL_CALCULADO', 'CD_PAI_VIRTUAL': 'CD_CONTA'}, inplace=True)
        soma_filhos['CD_CONTA_QTD_DIGITOS'] = nivel_filho - 2 
        
        # Merge para comparação
        keys = ['CNPJ_CIA', 'DT_REFER', 'CD_CONTA']
        df.set_index(keys, inplace=True)
        soma_filhos.set_index(keys, inplace=True)
        
        # A. UPDATE (SAFE MODE): O pai já existe?
        idx_comum = df.index.intersection(soma_filhos.index)
        if not idx_comum.empty:
            vals_calc = soma_filhos.loc[idx_comum, 'VL_CALCULADO']
            vals_orig = df.loc[idx_comum, 'VL_CONTA_TRATADO']
            
            # Lógica de Proteção:
            # Só atualizamos se o original for ZERO (buraco).
            # Se o original tiver valor, confiamos nele (Auditado) e ignoramos a soma dos filhos.
            mask_fix = (vals_orig.abs() < 0.01) & (vals_calc.abs() > 0.01)
            idx_update = idx_comum[mask_fix]
            
            if not idx_update.empty:
                df.loc[idx_update, 'VL_CONTA_TRATADO'] = vals_calc.loc[idx_update]
                total_reconstruido += len(idx_update)
            
            # Apenas para log: quantos pais protegemos de serem sobrescritos?
            # (Aqui cairiam os casos onde a soma analítica não bate com o sintético)
            mask_protected = (vals_orig.abs() > 0.01) & (~np.isclose(vals_calc, vals_orig, atol=1.0))
            total_protegido += mask_protected.sum()

        # B. INSERT: O pai não existe? (Linhas fantasmas)
        idx_novos = soma_filhos.index.difference(df.index)
        if not idx_novos.empty:
            novos = soma_filhos.loc[idx_novos].reset_index()
            novos.rename(columns={'VL_CALCULADO': 'VL_CONTA_TRATADO'}, inplace=True)
            df.reset_index(inplace=True)
            df = pd.concat([df, novos], ignore_index=True)
            total_reconstruido += len(idx_novos)
        else:
            df.reset_index(inplace=True)

    print(f"✅ Reconstrução Vertical DFC concluída.")
    print(f"   -> Lacunas preenchidas (Pais criados/fixados): {total_reconstruido}")
    print(f"   -> Valores Originais Mantidos (Divergência Pai vs Filhos): {total_protegido}")
    return df

In [11]:
# ==============================================================================
# 2. PADRONIZAÇÃO E VALIDAÇÃO HORIZONTAL (V13 - FINAL)
# ==============================================================================
def finalizar_dfc_silver_final(df_curado, df_original_completo):
    print("💎 INICIANDO FINALIZAÇÃO DFC V13 (HORIZONTAL CHECK)...")
    df_final = df_curado.copy()
    
    COLUNAS_ALVO = [
        "CNPJ_CIA", "SETOR_ATIV", "DT_REFER", "DT_REFER_TRATADO", "DT_REFER_ANO", 
        "VERSAO", "DENOM_CIA", "CD_CVM", "GRUPO_DFP_TRATADO", 
        "DT_FIM_EXERC_TRATADO", "DT_FIM_EXERC_ANO", 
        "CD_CONTA", "CD_CONTA_QTD_DIGITOS", 
        "DS_CONTA", "DS_CONTA_REPORTADA",       
        "FLAG_NORMALIZACAO", "FLAG_RECONSTRUCAO", 
        "STATUS_DFC", # <--- O veredito matemático
        "CONTA_NOME_COMPLETO", "VL_CONTA_TRATADO", "ST_CONTA_FIXA", "ST_CONTA_FIXA_REPORTADA",  
        "IS_LEAF", "DS_NIVEL_1", "DS_NIVEL_2", "DS_NIVEL_3", "DS_NIVEL_4", "DS_NIVEL_5", 
        "_origem_tabela"
    ]
    
    # 1. Traceability (Resgate de metadados originais)
    df_orig = df_original_completo.sort_values('VERSAO', ascending=False).drop_duplicates(['CNPJ_CIA', 'DT_REFER', 'CD_CONTA']).copy()
    df_orig.rename(columns={'DS_CONTA': 'DS_CONTA_REPORTADA', 'ST_CONTA_FIXA': 'ST_CONTA_FIXA_REPORTADA'}, inplace=True)
    df_final = pd.merge(df_final, df_orig[['CNPJ_CIA', 'DT_REFER', 'CD_CONTA', 'DS_CONTA_REPORTADA', 'ST_CONTA_FIXA_REPORTADA']], on=['CNPJ_CIA', 'DT_REFER', 'CD_CONTA'], how='left')
    df_final['FLAG_RECONSTRUCAO'] = df_final['DS_CONTA_REPORTADA'].isna()

    # 2. Golden Map (Padronização de Nomes)
    df_nomes = df_original_completo[['CD_CONTA', 'DS_CONTA']].dropna()
    df_nomes['DS_CONTA'] = df_nomes['DS_CONTA'].str.strip().str.upper()
    ranking = df_nomes.groupby(['CD_CONTA', 'DS_CONTA']).size().reset_index(name='FREQ').sort_values(['CD_CONTA', 'FREQ'], ascending=[True, False])
    golden_map = ranking.drop_duplicates('CD_CONTA').set_index('CD_CONTA')['DS_CONTA'].to_dict()
    
    # Forçamos nomes padrão para as contas-chave da equação
    golden_map['6.01'] = 'CAIXA LÍQUIDO ATIVIDADES OPERACIONAIS'
    golden_map['6.02'] = 'CAIXA LÍQUIDO ATIVIDADES DE INVESTIMENTO'
    golden_map['6.03'] = 'CAIXA LÍQUIDO ATIVIDADES DE FINANCIAMENTO'
    golden_map['6.04'] = 'VARIAÇÃO CAMBIAL SOBRE CAIXA E EQUIVALENTES'
    golden_map['6.05'] = 'AUMENTO (REDUÇÃO) DE CAIXA E EQUIVALENTES'
    
    df_final['DS_CONTA'] = df_final['CD_CONTA'].map(golden_map).fillna(df_final['DS_CONTA_REPORTADA']).fillna("CONTA RECONSTRUÍDA - " + df_final['CD_CONTA'])
    
    # === GOLDEN MAP --- ST_CONTA_FIXA ===
    df_fixa = df_original_completo[['CD_CONTA', 'ST_CONTA_FIXA']].dropna()
    ranking_fixa = df_fixa.groupby(['CD_CONTA', 'ST_CONTA_FIXA']).size().reset_index(name='FREQ')
    ranking_fixa.sort_values(['CD_CONTA', 'FREQ'], ascending=[True, False], inplace=True)
    golden_map_fixa = ranking_fixa.drop_duplicates(subset=['CD_CONTA']).set_index('CD_CONTA')['ST_CONTA_FIXA'].to_dict()
    df_final['ST_CONTA_FIXA'] = df_final['CD_CONTA'].map(golden_map_fixa).fillna('S')
    df_final['FLAG_NORMALIZACAO'] = (df_final['DS_CONTA'] != df_final['DS_CONTA_REPORTADA'].fillna("").str.upper()) & (~df_final['FLAG_RECONSTRUCAO'])

    # 3. DATA QUALITY (Math Check Horizontal)
    # Equação: 6.01 + 6.02 + 6.03 + 6.04 = 6.05
    print("   -> Calculando Equação Fundamental (6.01 a 6.04 = 6.05)...")
    contas_check = ['6.01', '6.02', '6.03', '6.04', '6.05']
    
    check = df_final[df_final['CD_CONTA'].isin(contas_check)].pivot_table(
        index=['CNPJ_CIA', 'DT_REFER'], columns='CD_CONTA', values='VL_CONTA_TRATADO', aggfunc='sum'
    ).fillna(0)
    
    # Garante que todas colunas existam (para evitar erro se ninguém reportou 6.04 no lote)
    for c in contas_check: 
        if c not in check.columns: check[c] = 0
            
    # Cálculo da divergência
    check['SOMA_MOVIMENTOS'] = check['6.01'] + check['6.02'] + check['6.03'] + check['6.04']
    check['DIFF'] = check['SOMA_MOVIMENTOS'] - check['6.05']
    
    # Definição do Status
    check['STATUS_DFC'] = np.where(check['DIFF'].abs() <= 1.0, 'OK', 'DIVERGENTE')
    
    # Se a empresa não tem a conta 6.05, é um erro estrutural grave
    check.loc[check['6.05'] == 0, 'STATUS_DFC'] = 'SEM_TOTALIZADOR'

    df_final = pd.merge(df_final, check[['STATUS_DFC']], on=['CNPJ_CIA', 'DT_REFER'], how='left')
    df_final['STATUS_DFC'] = df_final['STATUS_DFC'].fillna('OK') # Assumimos OK para empresas que não entraram no pivot (casos raros)

    # 4. Enriquecimento e Hierarquia
    cols_meta = ['CNPJ_CIA', 'DT_REFER', 'VERSAO', 'DT_FIM_EXERC_TRATADO', 'DT_FIM_EXERC_ANO', 'CD_CVM', 'SETOR_ATIV']
    df_meta = df_original_completo[cols_meta].sort_values('VERSAO', ascending=False).drop_duplicates(['CNPJ_CIA', 'DT_REFER'])
    for c in cols_meta:
        if c in df_final.columns and c not in ['CNPJ_CIA', 'DT_REFER']: df_final.drop(columns=[c], inplace=True)
    df_final = pd.merge(df_final, df_meta, on=['CNPJ_CIA', 'DT_REFER'], how='left')
    
    df_final['GRUPO_DFP_TRATADO'] = 'DFC'
    for col in ['SETOR_ATIV', 'CD_CVM', 'DENOM_CIA']: df_final[col] = df_final.groupby('CNPJ_CIA')[col].ffill().bfill()
    
    # Hierarquia Visual
    for i in range(1, 6):
        if i == 1: 
            # DFC não tem conta raiz "6" reportada, criamos o label fixo
            df_final[f'DS_NIVEL_{i}'] = 'DEMONSTRAÇÃO DO FLUXO DE CAIXA' 
        else:
            cod = df_final['CD_CONTA'].apply(lambda x: ".".join(x.split('.')[:i]) if x.count('.') >= (i-1) else np.nan)
            df_final[f'DS_NIVEL_{i}'] = cod.map(golden_map)

    # 5. Finalização de Tipagem
    df_final['DT_REFER'] = pd.to_datetime(df_final['DT_REFER'])
    df_final['DT_REFER_TRATADO'] = df_final['DT_REFER']
    df_final['DT_REFER_ANO'] = df_final['DT_REFER'].dt.year
    df_final['VERSAO'] = df_final['VERSAO'].fillna(1).astype(int)
    df_final['CONTA_NOME_COMPLETO'] = df_final['CD_CONTA'] + ' - ' + df_final['DS_CONTA']
    df_final['_origem_tabela'] = 'layer_02_silver.n0_dfp_cia_aberta (Curated V13 DFC)'
    
    # Recalcula IS_LEAF com base na estrutura final
    df_final['TEMP_PAI'] = df_final['CD_CONTA'].astype(str).str.rsplit('.', n=1).str[0]
    pais = set(df_final['TEMP_PAI'].unique())
    df_final['IS_LEAF'] = ~df_final['CD_CONTA'].isin(pais)

    for col in COLUNAS_ALVO:
        if col not in df_final.columns: df_final[col] = None
            
    return df_final[COLUNAS_ALVO]

### 4.2 Execução do Pipeline

In [12]:
# 1. Fase de Cura
print("\n>>> PASSO 1: RECONSTRUÇÃO NUMÉRICA DOS DADOS OBTIDOS DO BANCO DE DADOS")
df_curado = executar_sanity_check_v11_dfc_safe(df_dfc)

# 2. Fase de Enriquecimento (Seu script V13)
# (Assume-se que finalizar_dfc_silver_traceability_v13_fix já foi definido anteriormente)
print("\n>>> PASSO 2: PADRONIZAÇÃO E ENRIQUECIMENTO (V13)")
df_silver_final = finalizar_dfc_silver_final(df_curado, df_dfc)



>>> PASSO 1: RECONSTRUÇÃO NUMÉRICA DOS DADOS OBTIDOS DO BANCO DE DADOS
🏗️ INICIANDO SANITY CHECK V11 (DFC SAFE MODE - VERTICAL CHECK)...
✅ Reconstrução Vertical DFC concluída.
   -> Lacunas preenchidas (Pais criados/fixados): 0
   -> Valores Originais Mantidos (Divergência Pai vs Filhos): 2180

>>> PASSO 2: PADRONIZAÇÃO E ENRIQUECIMENTO (V13)
💎 INICIANDO FINALIZAÇÃO DFC V13 (HORIZONTAL CHECK)...
   -> Calculando Equação Fundamental (6.01 a 6.04 = 6.05)...


In [13]:
# 1. Filtro para achar a Engie 2021
print("🔍 AUDITORIA ESPECÍFICA (ENGIE 2021):")
filtro_engie = (
    (df_silver_final['DENOM_CIA'].str.contains('ENGIE', case=False, na=False)) & 
    (df_silver_final['DT_REFER_ANO'] == 2021)
)
cols_show = ['CNPJ_CIA', 'DENOM_CIA', 'DT_REFER', 'STATUS_DFC']
print(df_silver_final.loc[filtro_engie, cols_show].drop_duplicates())
# 2. Placar Geral de Qualidade (Horizontal)
print("\n📊 PLACAR GERAL (STATUS_DFC):")
print(df_silver_final[['CNPJ_CIA', 'DT_REFER', 'STATUS_DFC']].drop_duplicates()['STATUS_DFC'].value_counts())

# 3. Top 5 Maiores Divergências Horizontais (para você ver que não é só a Engie)
# (Recalculando a diferença aqui rapidinho para exibir)
pivot_check = df_silver_final[df_silver_final['CD_CONTA'].isin(['6.01','6.02','6.03','6.04','6.05'])].pivot_table(
    index=['DENOM_CIA', 'DT_REFER'], 
    columns='CD_CONTA', 
    values='VL_CONTA_TRATADO', 
    aggfunc='sum'
).fillna(0)

pivot_check['DIFF'] = (pivot_check.get('6.01',0) + pivot_check.get('6.02',0) + pivot_check.get('6.03',0) + pivot_check.get('6.04',0)) - pivot_check.get('6.05',0)
erros_reais = pivot_check[pivot_check['DIFF'].abs() > 1].sort_values('DIFF', key=abs, ascending=False)

print(f"\n🚨 TOP 5 DIVERGÊNCIAS ENCONTRADAS (Total de {len(erros_reais)} casos):")
print(erros_reais['DIFF'].head(5))

🔍 AUDITORIA ESPECÍFICA (ENGIE 2021):
                CNPJ_CIA                  DENOM_CIA   DT_REFER  STATUS_DFC
3370  02.474.103/0001-19  ENGIE BRASIL ENERGIA S.A. 2021-12-31  DIVERGENTE

📊 PLACAR GERAL (STATUS_DFC):
STATUS_DFC
OK            2179
DIVERGENTE       1
Name: count, dtype: int64

🚨 TOP 5 DIVERGÊNCIAS ENCONTRADAS (Total de 1 casos):
DENOM_CIA                  DT_REFER  
ENGIE BRASIL ENERGIA S.A.  2021-12-31   -828000.0
Name: DIFF, dtype: float64


In [14]:
df_silver_final.columns

Index(['CNPJ_CIA', 'SETOR_ATIV', 'DT_REFER', 'DT_REFER_TRATADO',
       'DT_REFER_ANO', 'VERSAO', 'DENOM_CIA', 'CD_CVM', 'GRUPO_DFP_TRATADO',
       'DT_FIM_EXERC_TRATADO', 'DT_FIM_EXERC_ANO', 'CD_CONTA',
       'CD_CONTA_QTD_DIGITOS', 'DS_CONTA', 'DS_CONTA_REPORTADA',
       'FLAG_NORMALIZACAO', 'FLAG_RECONSTRUCAO', 'STATUS_DFC',
       'CONTA_NOME_COMPLETO', 'VL_CONTA_TRATADO', 'ST_CONTA_FIXA',
       'ST_CONTA_FIXA_REPORTADA', 'IS_LEAF', 'DS_NIVEL_1', 'DS_NIVEL_2',
       'DS_NIVEL_3', 'DS_NIVEL_4', 'DS_NIVEL_5', '_origem_tabela'],
      dtype='object')

📋 Schema Alvo Final: Tabelas Harmonizadas (Golden Schema)

Abaixo está a definição da estrutura unificada para as tabelas da camada Silver (`BP`, `DRE`, `DFC`). O objetivo deste schema é permitir o empilhamento (`UNION ALL`) e a criação de uma dimensão temporal e de entidade consistentes para Business Intelligence.

| Grupo | Coluna | Tipo | Descrição / Racional |
| :--- | :--- | :--- | :--- |
| **Chaves** | `CNPJ_CIA` | Texto | Chave primária da entidade (CNPJ). |
| | `DT_REFER` | Data | Data de corte do balanço. Fundamental para *Time Intelligence*. |
| | `CD_CONTA` | Texto | Código contábil (Ex: `6.01`). É o ID único da linha no demonstrativo. |
| **Entidade** | `DENOM_CIA` | Texto | Razão social / Nome da empresa. |
| | `CD_CVM` | Int | Código CVM (usado para joins com dados de mercado/ações). |
| | `SETOR_ATIV` | Texto | Setor de atividade econômica na B3. |
| **Tempo** | `DT_REFER_TRATADO` | Date | Versão "Data Pura" (sem hora) do `DT_REFER`. |
| | `DT_REFER_ANO` | Int | Ano (ex: 2021, 2022) para filtros rápidos e particionamento. |
| | `DT_FIM_EXERC_TRATADO`| Date | Data de fim do exercício fiscal (útil para empresas com ano fiscal deslocado). |
| | `DT_FIM_EXERC_ANO` | Int | Ano do exercício fiscal. |
| | `VERSAO` | Int | Versão do documento entregue à CVM (1, 2...). |
| **Negócio** | `GRUPO_DFP_TRATADO` | Texto | Identificador do demonstrativo: `'BP'`, `'DRE'`, `'DFC'`. Chave do empilhamento. |
| | `VL_CONTA_TRATADO` | Float | Valor monetário unificado. Mesma escala e moeda para todas as linhas. |
| | `DS_CONTA` | Texto | Nome da conta **Curado/Padronizado** (Ex: "CAIXA E EQUIVALENTES"). |
| | `CONTA_NOME_COMPLETO` | Texto | Concatenação `CD_CONTA` + `DS_CONTA`. Ideal para eixos de gráficos. |
| | `CD_CONTA_QTD_DIGITOS`| Int | Nível hierárquico numérico da conta (1, 3, 5...). |
| | `ST_CONTA_FIXA` | Texto | `'S'` ou `'N'`. Indica se é uma conta fixa obrigatória da taxonomia CVM. |
| **Qualidade** | `DS_CONTA_REPORTADA` | Texto | Nome original vindo do CSV (Ex: "Variação monetária"). Vital para auditoria (Traceability). |
| **(Traceability)** | `ST_CONTA_FIXA_REPORTADA`| Texto | Status original da conta fixa vindo do arquivo bruto. |
| | `FLAG_NORMALIZACAO` | Bool | `True` se o nome da conta foi alterado/padronizado via *Golden Map*. |
| | `FLAG_RECONSTRUCAO` | Bool | `True` se o valor foi calculado via Python (*Safe Mode*) ou se a linha foi criada artificialmente. |
| | `STATUS_MATH` | Texto | Status da validação contábil: `'OK'`, `'DIVERGENTE'` ou `'DESBALANCEADO'`. |
| **Estrutura** | `IS_LEAF` | Bool | Indica se é o último nível (analítico). Útil para evitar soma duplicada em BI. |
| | `DS_NIVEL_1` ... `5` | Texto | Colunas de hierarquia para *Drill-Down* visual e matrizes. |
| **Metadados** | `_origem_tabela` | Texto | Nome da tabela da camada Bronze de onde o dado foi extraído. |

In [15]:
def padronizar_schema_silver(df_final, demonstrativo):
    """
    Recebe o DataFrame final de qualquer demonstrativo (BP, DRE, DFC) 
    e garante que ele tenha as 26 colunas do Golden Schema.
    """
    # 1. Renomeação de Colunas Específicas para o Padrão
    # Se existir STATUS_DFC (no caso da DFC), vira STATUS_MATH
    if 'STATUS_DFC' in df_final.columns:
        df_final.rename(columns={'STATUS_DFC': 'STATUS_MATH'}, inplace=True)
    
    # Se existir validação do BP (Active - Passive), vira STATUS_MATH
    if 'STATUS_BALANCO' in df_final.columns:
        df_final.rename(columns={'STATUS_BALANCO': 'STATUS_MATH'}, inplace=True)
        
    # Se não tiver STATUS_MATH (ex: DRE que não implementamos checagem horizontal ainda), cria OK
    if 'STATUS_MATH' not in df_final.columns:
        df_final['STATUS_MATH'] = 'NAO_APLICAVEL' # Ou 'OK' se confiar cegamente

    # 2. Garantia de Colunas de Qualidade (Para BP que estava sem)
    cols_qualidade = ['DS_CONTA_REPORTADA', 'ST_CONTA_FIXA_REPORTADA', 'FLAG_NORMALIZACAO', 'FLAG_RECONSTRUCAO']
    for col in cols_qualidade:
        if col not in df_final.columns:
            # Se não tem, é porque não fizemos o merge com o original nesse script.
            # Idealmente, o script do BP deve ser atualizado para fazer o merge igual ao da DFC.
            # Por enquanto, preenchemos com Nulo/False para não quebrar o schema.
            df_final[col] = None 
            if 'FLAG' in col: df_final[col] = False

    # 3. Criação de Colunas Calculadas Faltantes
    if 'CONTA_NOME_COMPLETO' not in df_final.columns:
        df_final['CONTA_NOME_COMPLETO'] = df_final['CD_CONTA'].astype(str) + ' - ' + df_final['DS_CONTA'].astype(str)

    # 4. Seleção e Ordenação Final
    COLUNAS_ALVO = [
        "CNPJ_CIA", "SETOR_ATIV", "DT_REFER", "DT_REFER_TRATADO", "DT_REFER_ANO", 
        "VERSAO", "DENOM_CIA", "CD_CVM", "GRUPO_DFP_TRATADO", 
        "DT_FIM_EXERC_TRATADO", "DT_FIM_EXERC_ANO", 
        "CD_CONTA", "CD_CONTA_QTD_DIGITOS", 
        "DS_CONTA", "DS_CONTA_REPORTADA",       
        "FLAG_NORMALIZACAO", "FLAG_RECONSTRUCAO", "STATUS_MATH",
        "CONTA_NOME_COMPLETO", "VL_CONTA_TRATADO", "ST_CONTA_FIXA", "ST_CONTA_FIXA_REPORTADA",  
        "IS_LEAF", "DS_NIVEL_1", "DS_NIVEL_2", "DS_NIVEL_3", "DS_NIVEL_4", "DS_NIVEL_5", 
        "_origem_tabela"
    ]
    
    # Garante que todas existam
    for col in COLUNAS_ALVO:
        if col not in df_final.columns:
            df_final[col] = None
            
    return df_final[COLUNAS_ALVO]

In [16]:
df_dfc_ready = padronizar_schema_silver(df_silver_final, 'DFC')

In [17]:
# Check Visual
df_dfc_ready.columns

Index(['CNPJ_CIA', 'SETOR_ATIV', 'DT_REFER', 'DT_REFER_TRATADO',
       'DT_REFER_ANO', 'VERSAO', 'DENOM_CIA', 'CD_CVM', 'GRUPO_DFP_TRATADO',
       'DT_FIM_EXERC_TRATADO', 'DT_FIM_EXERC_ANO', 'CD_CONTA',
       'CD_CONTA_QTD_DIGITOS', 'DS_CONTA', 'DS_CONTA_REPORTADA',
       'FLAG_NORMALIZACAO', 'FLAG_RECONSTRUCAO', 'STATUS_MATH',
       'CONTA_NOME_COMPLETO', 'VL_CONTA_TRATADO', 'ST_CONTA_FIXA',
       'ST_CONTA_FIXA_REPORTADA', 'IS_LEAF', 'DS_NIVEL_1', 'DS_NIVEL_2',
       'DS_NIVEL_3', 'DS_NIVEL_4', 'DS_NIVEL_5', '_origem_tabela'],
      dtype='object')

## 6. Escrita na Camada Silver — `n1_dfp_cia_aberta_dfc`

In [18]:
df_dfc_ready.to_sql(
            name='n1_dfp_cia_aberta_dfc',
            schema='layer_02_silver',
            con=engine,
            if_exists='replace',
            index=False,
            chunksize=10000, # Otimização para escrita em lotes
            method='multi'   # Otimização para insert múltiplo (dependendo do DB, pode ser removido)
        )

88876